In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#this is done in order to import tensorflow then keras and later we will use keras's layer ssystem in our model.

In [6]:

i_s = (224,224)
b_s = 32
#these are image size and batch(number of images in one epoch) size respectively.
dir = r"E:\Downloads\Telegram Desktop\plant_disease\all\Crop Diseases\sugarcane"
# now creating treaning set.
train = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "training",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
    
val = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "validation",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
#this is the validation set
#now time for dividing validation set into val and test sets
valbatches = tf.data.experimental.cardinality(val)
test = val.take(valbatches//2)
val= val.skip(valbatches//2)
class_names = train.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train).numpy())
print("Val batches:", tf.data.experimental.cardinality(val).numpy())
print("Test batches:", tf.data.experimental.cardinality(test).numpy())
# it is better to shuffle trianing set before augmentation
train = train.shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
#previous line means it shuffle in batch of 1000 and cpu automatically loads the next batch while gpu is training model by using prefetch function
val   = val.prefetch(buffer_size=tf.data.AUTOTUNE)
test  = test.prefetch(buffer_size=tf.data.AUTOTUNE)
#now its time for augmentation
aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomContrast(0.2),
])
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import optimizers , models, layers
num = len(class_names)

base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base.trainable = False
from keras.callbacks import EarlyStopping , ReduceLROnPlateau , ModelCheckpoint
inputs = layers.Input((224, 224, 3))
x = aug(inputs)

x = base(x, training = False)
#this is the phase of augmentation and normalization like max norm

x = layers.GlobalAveragePooling2D()(x) #this is pooling
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num, activation = "softmax")(x)
model = models.Model(inputs, outputs)
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3),
    ModelCheckpoint("models/head_stage.h5", save_best_only=True)
]
history_head = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=callbacks
)


Found 200 files belonging to 2 classes.
Using 160 files for training.
Found 200 files belonging to 2 classes.
Using 40 files for validation.
Classes: ['Sugarcane_Bacterial Blight', 'Sugarcane_Red Rot']
Train batches: 5
Val batches: 1
Test batches: 1
Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5731 - loss: 1.0098

5/5 ━━━━━━━━━━━━━━━━━━━━ 43s 3s/step - accuracy: 0.6313 - loss: 0.9065 - val_accuracy: 0.7500 - val_loss: 0.6463 - learning_rate: 0.0010
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 979ms/step - accuracy: 0.7570 - loss: 0.6618

5/5 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.7563 - loss: 0.6577 - val_accuracy: 0.6250 - val_loss: 0.4958 - learning_rate: 0.0010
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.7625 - loss: 0.7647 - val_accuracy: 0.6250 - val_loss: 0.7122 - learning_rate: 0.0010
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.8438 - loss: 0.3021 - val_accuracy: 0.5000 - val_loss: 0.7853 - learning_rate: 0.0010
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.8438 - loss: 0.3390 - val_accuracy: 0.6250 - val_loss: 0.7757 - learning_rate: 3.0000e-04


In [7]:

base.trainable = True


for layer in base.layers[:-30]:
    layer.trainable = False


model.compile(
    optimizer=optimizers.Adam(1e-5), 
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


history_finetune = model.fit(train, validation_data=val, epochs=20, callbacks=callbacks)

Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.7125 - loss: 0.7032 - val_accuracy: 0.6250 - val_loss: 0.9212 - learning_rate: 1.0000e-05
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step - accuracy: 0.7812 - loss: 0.6073 - val_accuracy: 0.7500 - val_loss: 0.6885 - learning_rate: 1.0000e-05
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - accuracy: 0.7500 - loss: 0.6113 - val_accuracy: 0.7500 - val_loss: 0.5676 - learning_rate: 3.0000e-06


In [8]:
test_loss, test_accuracy = model.evaluate(test)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6250 - loss: 0.7403
Test Loss: 0.7403
Test Accuracy: 62.50%


In [9]:
model.save("sugarcane.keras")